# **1. PRE-PROCESSING DATA**

1. Kiểm tra giá trị trùng lặp
2. Xử lý giá trị thiếu
3. Làm sạch và Chuẩn hóa dữ liệu
4. Đồng bộ hóa tên cột
5. Feature engineering


## **Pre-processinng**

## Import thư viện cần thiết

In [ ]:
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Đọc dữ liệu

In [ ]:
df = pd.read_csv("..\\data\\raw\\YoutubeCrawl_raw.csv")
print("Tổng quan dữ liệu:")
df.info()

Tổng quan dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10852 entries, 0 to 10851
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   video_id             10852 non-null  object
 1   title                10852 non-null  object
 2   description          9522 non-null   object
 3   channel_id           10852 non-null  object
 4   channel_title        10852 non-null  object
 5   category_id          10852 non-null  int64 
 6   publish_date         10852 non-null  object
 7   duration_iso         10852 non-null  object
 8   view_count           10852 non-null  int64 
 9   like_count           10852 non-null  int64 
 10  comment_count        10852 non-null  int64 
 11  favorite_count       10852 non-null  int64 
 12  subscriber_count     10852 non-null  int64 
 13  channel_view_count   10852 non-null  int64 
 14  channel_video_count  10852 non-null  int64 
 15  channel_created_at   10832 non-nul

In [ ]:
df.head(5)

,video_id,title,description,channel_id,channel_title,category_id,publish_date,duration_iso,view_count,like_count,comment_count,favorite_count,subscriber_count,channel_view_count,channel_video_count,channel_created_at,tags,caption_status,topics
0,8uiZC0l4Ajw,Learn GO Fast: Full Tutorial,This is a full tutorial on learning Golang! Fr...,UCr3L56DsMxhNtHqtm2AIsMw,Alex Mux,28,2023-09-04T13:47:22Z,PT1H7M53S,875767,28661,780,0,21800,1061057,16,2023-02-19T13:38:58.930641Z,NaN,False,https://en.wikipedia.org/wiki/Knowledge|https:...
1,c-P5R0aMylM,Ken Thompson: Why did we create Golang?,Ken Thompson answers a question about the reas...,UCZoBjunTdzzT-HVI-Tb4xiw,Mastery Learning,28,2024-01-24T19:20:12Z,PT31S,187932,2513,159,0,8560,2809707,15,2018-09-16T17:33:24Z,go|go programming|golang|golang tutorial|golan...,False,https://en.wikipedia.org/wiki/Knowledge|https:...
2,un6ZyFkqFKo,Go Programming – Golang Course with Bonus Proj...,Learn the Go programming language in this full...,UC8butISFwT-Wl7EV0hUK0BQ,freeCodeCamp.org,27,2023-05-11T13:41:01Z,PT9H32M48S,1270394,23500,524,0,11300000,918891399,1979,2014-12-16T21:18:48Z,NaN,False,https://en.wikipedia.org/wiki/Knowledge|https:...
3,XCZWyN9ZbEQ,Full Golang Tutorial - Learn Go by Building a ...,A Complete Go Crash Course for Beginners to le...,UCdngmbVKX1Tgre699-XLlUA,TechWorld with Nana,27,2024-10-14T13:22:24Z,PT1H34M56S,184015,4690,151,0,1360000,74801846,141,2019-10-06T08:50:17Z,golang|programming|go tutorial|golang tutorial...,False,https://en.wikipedia.org/wiki/Knowledge|https:...
4,6gwF8mG3UUY,Why I Use Golang In 2024,"Recorded live on twitch, GET IN \r\n\r\nhttps:...",UCUyeluBRhGPCW4rPe_UvBZQ,ThePrimeTime,28,2024-02-14T14:00:04Z,PT9M21S,444111,10504,715,0,937000,280822810,1546,2021-03-05T17:02:28.224517Z,programming|software engineer|software enginee...,False,https://en.wikipedia.org/wiki/Knowledge


## Bước 1: Kiểm tra giá trị trùng lặp

Để kiểm soát *chất lượng* và *số lượng* dữ liệu một cách chặt chẽ hơn, việc đầu tiên cần thực hiện là kiểm tra các bản ghi trùng lặp trong dataset.

Do API có thể lấy về các bản ghi bị lặp lại (ví dụ: do thực hiện nhiều request trong cùng một khoảng thời gian, hoặc API trả về dữ liệu có timestamp trùng nhau), nên việc kiểm tra trùng lặp là bước quan trọng để đảm bảo chất lượng dữ liệu đầu vào.

Việc xử lý trùng lặp ngay từ đầu giúp tránh sai lệch khi phân tích, đồng thời giúp hệ thống quản lý dữ liệu hiệu quả hơn trong các bước tiếp theo.

In [ ]:
print(f"Tổng số video: {len(df)}")
print(f"Video có ids độc nhất: {df['video_id'].nunique()}")
print(f"Video có ids bị trùng: {len(df) - df['video_id'].nunique()}")

Tổng số video: 10852
Video có ids độc nhất: 10660
Video có ids bị trùng: 192


Dữ liệu có xảy ra trùng lặp video ID, với số lượng video bị trùng là 192 video.

==> Cần loại bỏ các bản ghi trùng lặp này để đảm bảo tính duy nhất của mỗi video trong dataset.

In [ ]:
# Loại bỏ các bản ghi trùng lặp dựa trên video_id
df = df.drop_duplicates(subset=['video_id'])
df = df.reset_index(drop=True)
df.info()
print("Sau khi loại bỏ trùng lặp:")
print(f"Tổng số video: {len(df)}")
print(f"Video có ids độc nhất: {df['video_id'].nunique()}")
print(f"Video có ids bị trùng: {len(df) - df['video_id'].nunique()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10660 entries, 0 to 10659
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   video_id             10660 non-null  object
 1   title                10660 non-null  object
 2   description          9355 non-null   object
 3   channel_id           10660 non-null  object
 4   channel_title        10660 non-null  object
 5   category_id          10660 non-null  int64 
 6   publish_date         10660 non-null  object
 7   duration_iso         10660 non-null  object
 8   view_count           10660 non-null  int64 
 9   like_count           10660 non-null  int64 
 10  comment_count        10660 non-null  int64 
 11  favorite_count       10660 non-null  int64 
 12  subscriber_count     10660 non-null  int64 
 13  channel_view_count   10660 non-null  int64 
 14  channel_video_count  10660 non-null  int64 
 15  channel_created_at   10640 non-null  object
 16  tags

## Bước 2: Xử lý giá trị thiếu

Dựa trên kết quả `df.info()` ở bước đọc dữ liệu, ta đã quan sát sơ bộ rằng dataset có xuất hiện giá trị thiếu ở một số cột.

Để có cái nhìn cụ thể và chính xác hơn, ta sẽ tiến hành:
- Thống kê số lượng giá trị thiếu trên từng cột.
- Xác định tỷ lệ phần trăm giá trị thiếu so với tổng số bản ghi.
- Đề xuất hướng xử lý phù hợp cho từng loại cột.

In [ ]:
# Kiểm tra giá trị thiếu
missing_values = df.isnull().sum()
print("Giá trị thiếu trong mỗi cột:")
print(missing_values[missing_values > 0])

# Tính tỷ lệ phần trăm giá trị thiếu
missing_percentage = (missing_values / len(df)) * 100
print("Tỷ lệ phần trăm giá trị thiếu trong mỗi cột:")
print(missing_percentage[missing_percentage > 0])

Giá trị thiếu trong mỗi cột:
description           1305
channel_created_at      20
tags                  3876
topics                 875
dtype: int64
Tỷ lệ phần trăm giá trị thiếu trong mỗi cột:
description           12.242026
channel_created_at     0.187617
tags                  36.360225
topics                 8.208255
dtype: float64


Có 4 cột có giá trị thiếu:
- `description`
- `channel_created_at`
- `tags`
- `topics`

### **Xử lý `description`**

`description` là phần mô tả nội dung video do người tạo video tự cung cấp. Đây không phải là trường dữ liệu bắt buộc và việc thiếu mô tả không ảnh hưởng đến các phân tích định lượng chính của dataset.

Do đó, với các bản ghi bị thiếu giá trị ở cột `description`, ta lựa chọn điền giá trị rỗng ("") để giữ nguyên số lượng bản ghi và đảm bảo các xử lý văn bản (nếu thực hiện sau này) có thể nhận diện dễ dàng trường hợp mô tả trống.

In [ ]:
# Description
df['description'] = df['description'].fillna('')

### **Xử lý `channel_created_at`**

Đối với cột `channel_created_at`, kết quả thống kê cho thấy chỉ có 20 dòng bị thiếu (chiếm ~0.18% trên toàn tập dữ liệu). 

Nhận thấy thuộc tính này có thể khai thác trong phân tích. Do đó, thay vì gán giá trị mặc định hoặc nội suy (vốn không phù hợp với dữ liệu dạng thời gian có tính chất duy nhất cho từng kênh), ta quyết định **loại bỏ 20 dòng dữ liệu** thiếu ở trường này.

In [ ]:
df = df.dropna(subset=['channel_created_at'])

### **Xử lý `tags`**

`tags` là trường chứa các thẻ gắn với video, thường được sử dụng để mô tả chủ đề hoặc giúp thuật toán phân loại nội dung tốt hơn. Đây là một trong những loại dữ liệu có thể mang ý nghĩa khi phân tích chủ đề, phân cụm nội dung hoặc xây dựng mô hình gợi ý. Tuy nhiên, thống kê cho thấy cột này có tỷ lệ giá trị thiếu khá lớn.

Do mức độ thiếu nhiều và chưa có đủ thông tin để xây dựng phương án xử lý phù hợp (ví dụ: suy đoán dựa trên tiêu đề, mô tả hoặc nội dung video), ta tạm thời áp dụng cách tiếp cận an toàn là điền giá trị rỗng ("") cho các trường thiếu.

In [ ]:
df['tags'] = df['tags'].fillna('')

### **Xử lý `topics`**

Cột `topics` thể hiện các chủ đề mà video thuộc về, thường được gắn tự động bởi hệ thống phân loại nội dung. Trường thông tin này có thể hữu ích trong các phân tích về chủ đề, xu hướng hoặc nhóm nội dung. Tuy nhiên, thống kê cho thấy `topics` là một trong những cột có tỷ lệ giá trị thiếu tương đối, dù vẫn còn đủ dữ liệu để khai thác.

Ta lựa chọn phương án đơn giản và an toàn: điền giá trị rỗng ("") cho các bản ghi thiếu `topics`.

In [ ]:
# Topics
df['topics'] = df['topics'].fillna('')

## Bước 3: Làm sạch và Chuẩn hóa dữ liệu

Sau khi xử lý các vấn đề về trùng lặp và giá trị thiếu, bước tiếp theo là làm sạch và chuẩn hóa dữ liệu nhằm đảm bảo tính nhất quán, đồng nhất và dễ xử lý trong các bước phân tích sau.

Trong bước này, ta thực hiện:
- Chuẩn hóa kiểu dữ liệu (datetime, số nguyên, số thực, chuỗi văn bản, v.v.).
- Làm sạch các cột văn bản.
- Loại bỏ các thuộc tính lỗi/nhiễu/không có giá trị sử dụng.

Do tập dữ liệu bao gồm 19 thuộc tính, ta sẽ chia theo trường dữ liệu để **Làm sạch và Chuẩn hóa**:
- IDs: Các trường có chức năng định danh (`video_id`, `channel_id`, `category_id`).
- Numeric: (`view_count`, `like_count`, `comment_count`, ...).
- Text: (`title`, `description`, `tags`, `topics`, ...).
- Binary: True/False (`caption_status`)
- Datetime: (`published_at`, `channel_created_at`, ...).

### **Xử lý IDs**

Ta sẽ tiến hành kiểm tra **IDs** theo quy trình:
- Xem dữ liệu.
- Làm sạch - Chuẩn hóa dữ liệu.

In [ ]:
id_cols = ['video_id', 'channel_id', 'category_id']
print("Các cột ID:", id_cols)

Các cột ID: ['video_id', 'channel_id', 'category_id']


### `video_id`, `channel_id`

In [ ]:
print("10 giá trị khác nhau trong cột video_id:")
print(df['video_id'].unique()[:10])

print("\n")

print("10 giá trị khác nhau trong cột channel_id:")
print(df['channel_id'].unique()[:10])

10 giá trị khác nhau trong cột video_id:
['8uiZC0l4Ajw' 'c-P5R0aMylM' 'un6ZyFkqFKo' 'XCZWyN9ZbEQ' '6gwF8mG3UUY'
 'UEU4SzBjqrc' 'qT14b1pxtiI' 'uBkXiyLbaQE' 'lbPThhcfn10' 'x-IxAXuFvmU']


10 giá trị khác nhau trong cột channel_id:
['UCr3L56DsMxhNtHqtm2AIsMw' 'UCZoBjunTdzzT-HVI-Tb4xiw'
 'UC8butISFwT-Wl7EV0hUK0BQ' 'UCdngmbVKX1Tgre699-XLlUA'
 'UCUyeluBRhGPCW4rPe_UvBZQ' 'UCXzw-OdotBUcNA9yhuYQBwA'
 'UC1Zfv1Zrp1q5lKgBomzOyCA' 'UCqerkZelwpOiq56XXPmfZSQ'
 'UC7EVSn5inapL20oPSwAwEUg' 'UCB5jYf6goa67NOPQeEgE--w']



==>> Các cột `video_id` và `channel_id` dùng để định danh duy nhất cho từng video và kênh, giúp phân biệt các bản ghi trong dataset. Vì đây là dữ liệu định danh, không phục vụ trực tiếp cho phân tích định lượng hay mô hình hóa.

Vì vậy, ta sẽ tiến hành loại bỏ `video_id` và `channel_id` ra khỏi tập dữ liệu

In [ ]:
df.drop(columns=['video_id'], inplace=True)

df.drop(columns=['channel_id'], inplace=True)

### `category_id`

Khác với `video_id` và `channel_id` chỉ dùng để định danh, cột category_id mang ý nghĩa phân loại nội dung của video theo từng chủ đề (ví dụ: Âm nhạc, Giáo dục, Giải trí…) dựa trên [`videoCategories.list` của **YouTube Data API v3**](https://gist.github.com/dgp/1b24bf2961521bd75d6c) (công cụ nhóm sử dụng để thu thập data từ Youtube).

Đây là một đặc trưng quan trọng.

In [ ]:
print("10 giá trị khác nhau trong cột category_id:")
print(df['category_id'].unique()[:10])

10 giá trị khác nhau trong cột category_id:
[28 27 22 24 19 26 10 20 17  1]


==>> Do đây `category_id` được lưu dưới dạng `int64` nên ta sẽ xử lý chung với các **cột Numeric cols**.

### **Xử lý Numeric cols (Cột số)**

Ở bước này, các cột dạng số như `view_count`, `like_count`, `comment_count`,... sẽ được làm sạch và chuẩn hóa. Mục tiêu là đảm bảo các giá trị số:
- Đúng kiểu dữ liệu (int64 hoặc float64).
- Loại bỏ giá trị bất thường (kiểm tra và xử lý các outlier hoặc giá trị âm không hợp lý).

In [ ]:
numeric_cols = ['category_id', 'view_count', 'like_count', 'comment_count', 'favorite_count', 'subscriber_count', 'channel_view_count', 'channel_video_count']
print("Các cột numeric:", numeric_cols)

Các cột numeric: ['category_id', 'view_count', 'like_count', 'comment_count', 'favorite_count', 'subscriber_count', 'channel_view_count', 'channel_video_count']


**1. Kiểm tra dữ liệu không phải số (NaN)**

In [ ]:
#kiểm tra NaN
nan_counts = df[numeric_cols].isna().sum()
print("Số lượng giá trị NaN trong các cột numeric:")
print(nan_counts)

Số lượng giá trị NaN trong các cột numeric:
category_id            0
view_count             0
like_count             0
comment_count          0
favorite_count         0
subscriber_count       0
channel_view_count     0
channel_video_count    0
dtype: int64


==>> Không có giá trị NaN cần xử lý.

In [ ]:
# Kiểm tra giá trị Numeric
print("Mô tả thống kê các cột numeric:")
print(df[numeric_cols].describe())

Mô tả thống kê các cột numeric:
        category_id    view_count    like_count  comment_count  \
count  10640.000000  1.064000e+04  1.064000e+04   10640.000000   
mean      25.956203  8.256731e+05  1.602765e+04     316.635620   
std        3.047014  9.169104e+06  1.223098e+05    8026.831253   
min        1.000000  0.000000e+00  0.000000e+00       0.000000   
25%       26.000000  4.690000e+02  9.000000e+00       0.000000   
50%       27.000000  1.015450e+04  2.140000e+02       8.000000   
75%       27.000000  1.206965e+05  2.835500e+03      70.000000   
max       29.000000  4.622285e+08  5.346957e+06  810239.000000   

       favorite_count  subscriber_count  channel_view_count  \
count         10640.0      1.064000e+04        1.064000e+04   
mean              0.0      8.719486e+05        3.270919e+08   
std               0.0      4.492815e+06        4.032213e+09   
min               0.0      0.000000e+00        0.000000e+00   
25%               0.0      1.050000e+03        1.394045e+0

==>> Cột `favorite_count` = 0 ở tất cả dòng dữ liệu. 

`favorite_count` là số lượt thêm vào "Yêu thích". Hiện nay API YouTube đã ngừng hỗ trợ công khai, nên cột này thường vô nghĩa.

Và giá trị không thay đổi (tất cả bằng 0), cột này không cung cấp thông tin gì hữu ích cho phân tích hay mô hình hóa.

Do đó, cột này có thể loại bỏ khỏi dataset để tránh dư thừa dữ liệu và giảm độ phức tạp.

In [ ]:
df.drop(columns=['favorite_count'], inplace=True)

### **Xử lý Category/Text cols (Cột chữ)**

Các cột chữ bao gồm các cột văn bản (text) như `title`, `description`, `tags`, `topics`. Nhóm này cần được làm sạch và chuẩn hóa để đảm bảo dữ liệu nhất quán và dễ khai thác.

Các bước xử lý chính:
- Làm sạch cột văn bản (Text).
- Kiểm tra độ trùng lặp và thống nhất.
- Vector hóa văn bản & Tính Cosine Similarity.

**Xử lý `title`, `description`**

In [ ]:
print("5 giá trị khác nhau trong cột title trước khi làm sạch:")
print(df['title'].unique()[:5])

print("\n")

print("5 giá trị khác nhau trong cột description trước khi làm sạch:")
print(df['description'].unique()[:5])

5 giá trị khác nhau trong cột title trước khi làm sạch:
['Learn GO Fast: Full Tutorial' 'Ken Thompson: Why did we create Golang?'
 'Go Programming – Golang Course with Bonus Projects'
 'Full Golang Tutorial - Learn Go by Building a TodoList App'
 'Why I Use Golang In 2024']


5 giá trị khác nhau trong cột description trước khi làm sạch:


['This is a full tutorial on learning Golang! From start to finish in less than an hour, including a full demo of how to build an api in Go. No fluff, just what you need to know.\r\n\r\n0:00 Introduction to Golang\r\n6:25 Constants Variables and Basic Data Types\r\n13:14 Functions and Control Structures\r\n19:30 Arrays, Slices, Maps and Loops\r\n26:36 Strings, Runes and Bytes\r\n31:06 Structs and Interfaces\r\n35:18 Pointers\r\n40:06 Goroutines\r\n47:10 Channels\r\n52:56 Generics\r\n55:47 Building an API!'
 'Ken Thompson answers a question about the reason behind the creation of Golang at Google I/O 2012.'
 "Learn the Go programming language in this full course for beginners. You'll practice writing performant, idiomatic Go with these hands-on lessons and challenges.\r\n\r\n💻 Code: https://github.com/bootdotdev/fcc-learn-golang-assets\r\n💻 Follow along interactively on Boot.dev: https://boot.dev/learn/learn-golang\r\n\r\n✏️ Course created by @bootdotdev \r\n🐦 Follow Lane on Twitter: ht

In [ ]:
def preprocess_text(text):
    if pd.isna(text):
        return []
    # lowercase + loại ký tự đặc biệt
    text = re.sub(r'[^a-z0-9\s]', '', text.lower())
    return text.split()

df['title'] = df['title'].astype(str)
df['description'] = df['description'].astype(str)
df['title'] = df['title'].apply(lambda x: ' '.join(preprocess_text(x)))
df['description'] = df['description'].apply(lambda x: ' '.join(preprocess_text(x)))

In [ ]:
print("5 giá trị khác nhau trong cột title sau khi làm sạch:")
print(df['title'].unique()[:5])

print("\n")

print("5 giá trị khác nhau trong cột description sau khi làm sạch:")
print(df['description'].unique()[:5])

5 giá trị khác nhau trong cột title sau khi làm sạch:
['learn go fast full tutorial' 'ken thompson why did we create golang'
 'go programming golang course with bonus projects'
 'full golang tutorial learn go by building a todolist app'
 'why i use golang in 2024']


5 giá trị khác nhau trong cột description sau khi làm sạch:
['this is a full tutorial on learning golang from start to finish in less than an hour including a full demo of how to build an api in go no fluff just what you need to know 000 introduction to golang 625 constants variables and basic data types 1314 functions and control structures 1930 arrays slices maps and loops 2636 strings runes and bytes 3106 structs and interfaces 3518 pointers 4006 goroutines 4710 channels 5256 generics 5547 building an api'
 'ken thompson answers a question about the reason behind the creation of golang at google io 2012'
 'learn the go programming language in this full course for beginners youll practice writing performant idiomatic go 

**Xử lý `topics`**

In [ ]:
def process_topics_safe(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        urls = x
    else:
        # giả sử x là string, có thể chứa nhiều URL phân tách bởi '|'
        urls = x.split('|')
    result = []
    for url in urls:
        if isinstance(url, str):
            # lấy phần cuối URL làm tên topic, loại bỏ '_' thành space
            result.append(url.split('/')[-1].replace('_', ' '))
    return result

df['topics'] = df['topics'].apply(process_topics_safe)

print(df['topics'].head(10))


0                              [Knowledge, Technology]
1                              [Knowledge, Technology]
2                              [Knowledge, Technology]
3                              [Knowledge, Technology]
4                                          [Knowledge]
5                              [Knowledge, Technology]
6                              [Knowledge, Technology]
7    [Hobby, Lifestyle (sociology), Puzzle video game]
8                              [Knowledge, Technology]
9                                         [Technology]
Name: topics, dtype: object


**Xử lý `tags`**

Các bước quy trình:
- Chuẩn hóa tags gốc.
- Phân tách dữ liệu.
- Vector hóa văn bản.
- Vector hóa văn bản & Tính Cosine Similarity.
- Gán tags.
- Kiểm tra và lưu giá trị đã xử lý.

In [ ]:
print("5 giá trị khác nhau trong cột tags trước khi làm sạch:")
print(df['tags'].unique()[:5])

5 giá trị khác nhau trong cột tags trước khi làm sạch:
[''
 'go|go programming|golang|golang tutorial|golang course|go course|go tutorial|rust|rust lang|rustlang|rust programming'
 'golang|programming|go tutorial|golang tutorial|golang for beginners|learn golang|go for beginners|go tutorial for beginners|golang tutorial for beginners|what is golang used for|golang slice|why golang|golang introduction|techworld with nana|why learn golang|what is golang good for|go lang|go programming|go|go crash course|learn go|golang crash course|go variables|loops in go|functions in golang|golang http server|golang http'
 'programming|software engineer|software engineering|developer|web design|web development|programmer humor'
 'video sharing|video|sharing|computer science|software engineer|silicon valley|computer programming|coding|learn to code|machine learning|artificial intelligence|cloud|python|javascript|how to code|Data scientist|AI developer|Machine Learning|how to learn go lang|how to learn g

**Bước 1:
Chuẩn hóa tags gốc:** Chuyển đổi dữ liệu tags thô từ chuỗi (string) sang danh sách (list), xử lý các dấu phân cách (|, ,) và loại bỏ giá trị rỗng.

In [ ]:
def process_tags(x):
    if pd.isna(x):
        return []
    # Nếu đã là list, giữ nguyên
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip() != '']
    # Nếu là string
    if isinstance(x, str):
        if x == '':
            return []
        if '|' in x:
            return [t.strip() for t in x.split('|') if t.strip() != '']
        elif ',' in x:
            return [t.strip() for t in x.split(',') if t.strip() != '']
        else:
            return [t.strip() for t in x.split() if t.strip() != '']
    return []

df['tags_clean'] = df['tags'].apply(process_tags)

**Bước 2: Phân tách dữ liệu:** Chia tập dữ liệu thành 2 nhóm: nhóm đã có tags (Available) và nhóm bị thiếu tags (Missing).

In [ ]:
df_reset = df.reset_index(drop=True)
available_idx = df_reset[df_reset['tags_clean'].map(len) > 0].index
missing_idx = df_reset[df_reset['tags_clean'].map(len) == 0].index

**Bước 3: Vector hóa văn bản & Tính Cosine Similarity:** Chuyển đổi title và description thành ma trận vector số học sử dụng kỹ thuật TF-IDF.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
corpus = (df_reset['title'] + ' ' + df_reset['description']).tolist()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

# hyperparameter để lấy top k tags
top_k = 5

tfidf_missing = tfidf_matrix[missing_idx]
tfidf_available = tfidf_matrix[available_idx]

# tính cosine similarity một lần choa tất cạ
sim_matrix = cosine_similarity(tfidf_missing, tfidf_available)

**Bước 4: Gán tags:** Gán tags cho video bị thiếu bằng cách tổng hợp tags từ 5 video giống nhất (Top-k Neighbors), ưu tiên lọc lấy các từ khóa có xuất hiện trong tiêu đề/mô tả của chính video đó.

In [ ]:
for i, idx in enumerate(missing_idx):
    sim = sim_matrix[i]
    topk_idx = available_idx[np.argsort(sim)[::-1][:top_k]]
    
    # gộp tags từ neighbors
    tags_suggested = set()
    for nidx in topk_idx:
        tags_suggested.update(df_reset.at[nidx, 'tags_clean'])
    
    # keywords từ title + description
    words_title_desc = set(df_reset.at[idx, 'title'].split() + df_reset.at[idx, 'description'].split())
    
    # intersection giữa neighbor tags và title/description
    relevant_tags = tags_suggested & words_title_desc
    if relevant_tags:
        df_reset.at[idx, 'tags_clean'] = list(relevant_tags)
    else:
        df_reset.at[idx, 'tags_clean'] = list(tags_suggested)

**Bước 5: Kiểm tra kết quả:** Kiểm tra kết quả trước khi gán vào df gốc

In [ ]:
pd.set_option('display.max_colwidth', None)
print(df_reset[['title', 'tags_clean']].head(10))

                                                                title  \
0                                         learn go fast full tutorial   
1                               ken thompson why did we create golang   
2                    go programming golang course with bonus projects   
3            full golang tutorial learn go by building a todolist app   
4                                            why i use golang in 2024   
5                                  hard truths before switching to go   
6            the best resources to learn golang if i could start over   
7      ladder a great teachnique to capture stones gogame baduk weiqi   
8  golang programming full course learn in 2025 golang golangtutorial   
9                  should you learn rust or golang programming shorts   

                                                                                                                                                                                                    

**Gán `tags_clean` đã xử lý thành công vào df gốc.**

In [ ]:
df["tags_clean"] = df_reset["tags_clean"]

### **Xử lý Binary cols (True/False)**

In [ ]:
# kiểm tra cột caption_status
print("Giá trị duy nhất trong cột caption_status:")
print(df['caption_status'].unique())

Giá trị duy nhất trong cột caption_status:
[False  True]


**Giá trị của thuộc tính đã là boolean nên giữ nguyên**

### **Xử lý Datetime cols (Cột thời gian)**

In [ ]:
df['publish_date'] = pd.to_datetime(df['publish_date'], format='mixed')
df['channel_created_at'] = pd.to_datetime(df['channel_created_at'], format='mixed')

**Xử lý `duration_iso`**

Chuẩn hóa duration từ chuẩn iso về giây bình thường

In [ ]:
#duration_iso to seconds
def iso_to_seconds(iso_str):
    pattern = re.compile(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?')
    match = pattern.match(iso_str)
    if not match:
        return 0
    hours = int(match.group(1)) if match.group(1) else 0
    minutes = int(match.group(2)) if match.group(2) else 0
    seconds = int(match.group(3)) if match.group(3) else 0
    total_seconds = hours * 3600 + minutes * 60 + seconds
    return total_seconds

df['duration_seconds'] = df['duration_iso'].apply(iso_to_seconds)

## Bước 4: Đồng bộ hóa tên cột

In [ ]:
# In ra tên các cột trong DataFrame trước khi đồng bộ hóa tên cột
print("Tên các cột trước khi đồng bộ hóa:")
print(df.columns)

Tên các cột trước khi đồng bộ hóa:
Index(['title', 'description', 'channel_title', 'category_id', 'publish_date',
       'duration_iso', 'view_count', 'like_count', 'comment_count',
       'subscriber_count', 'channel_view_count', 'channel_video_count',
       'channel_created_at', 'tags', 'caption_status', 'topics', 'tags_clean',
       'duration_seconds'],
      dtype='object')


In [ ]:
# đổi tên title -> video_title
df.rename(columns={'title': 'video_title'}, inplace=True)
# đổi tên description -> video_description
df.rename(columns={'description': 'video_description'}, inplace=True)

# xóa tags cột cũ
df.drop(columns=['tags'], inplace=True)
# đổi ten tags_clean -> video_tags
df.rename(columns={'tags_clean': 'video_tags'}, inplace=True)

# đổi tên publish_date -> video_publish_date
df.rename(columns={'publish_date': 'video_publish_date'}, inplace=True)
# đổi tên duration_seconds -> video_duration
df.rename(columns={'duration_seconds': 'video_duration'}, inplace=True)
# đổi tên view_count -> video_view_count
df.rename(columns={'view_count': 'video_view_count'}, inplace=True)
# đổi tên like_count -> video_like_count
df.rename(columns={'like_count': 'video_like_count'}, inplace=True)
# đổi tên comment_count -> video_comment_count
df.rename(columns={'comment_count': 'video_comment_count'}, inplace=True)
# đổi tên caption_status -> video_caption_status
df.rename(columns={'caption_status': 'video_caption_status'}, inplace=True)
# đổi tên topics -> video_topics
df.rename(columns={'topics': 'video_topics'}, inplace=True)

# đổi tên subcriber_count -> channel_subscriber_count
df.rename(columns={'subscriber_count': 'channel_subscriber_count'}, inplace=True)

In [ ]:
# Tên các cột sau khi đồng bộ hóa
print("Tên các cột sau khi đồng bộ hóa:")
print(df.columns)

Tên các cột sau khi đồng bộ hóa:
Index(['video_title', 'video_description', 'channel_title', 'category_id',
       'video_publish_date', 'duration_iso', 'video_view_count',
       'video_like_count', 'video_comment_count', 'channel_subscriber_count',
       'channel_view_count', 'channel_video_count', 'channel_created_at',
       'video_caption_status', 'video_topics', 'video_tags', 'video_duration'],
      dtype='object')


## Bước 5: Feature engineering

In [ ]:
# like_rate
df['video_like_rate'] = df['video_like_count'] / df['video_view_count'] 
df['video_comment_rate'] = df['video_comment_count'] / df['video_view_count']
# engagement rate
df['video_engagement_rate'] = (df['video_like_count'] + df['video_comment_count']) / df['video_view_count']

## Kiểm tra lại trước khi bước vào EDA

In [ ]:
df.head(5)

,video_title,video_description,channel_title,category_id,video_publish_date,duration_iso,video_view_count,video_like_count,video_comment_count,channel_subscriber_count,channel_view_count,channel_video_count,channel_created_at,video_caption_status,video_topics,video_tags,video_duration,video_like_rate,video_comment_rate,video_engagement_rate
0,learn go fast full tutorial,this is a full tutorial on learning golang from start to finish in less than an hour including a full demo of how to build an api in go no fluff just what you need to know 000 introduction to golang 625 constants variables and basic data types 1314 functions and control structures 1930 arrays slices maps and loops 2636 strings runes and bytes 3106 structs and interfaces 3518 pointers 4006 goroutines 4710 channels 5256 generics 5547 building an api,Alex Mux,28,2023-09-04 13:47:22+00:00,PT1H7M53S,875767,28661,780,21800,1061057,16,2023-02-19 13:38:58.930641+00:00,False,"[Knowledge, Technology]","[golang, go]",4073,0.032727,0.000891,0.033617
1,ken thompson why did we create golang,ken thompson answers a question about the reason behind the creation of golang at google io 2012,Mastery Learning,28,2024-01-24 19:20:12+00:00,PT31S,187932,2513,159,8560,2809707,15,2018-09-16 17:33:24+00:00,False,"[Knowledge, Technology]","[go, go programming, golang, golang tutorial, golang course, go course, go tutorial, rust, rust lang, rustlang, rust programming]",31,0.013372,0.000846,0.014218
2,go programming golang course with bonus projects,learn the go programming language in this full course for beginners youll practice writing performant idiomatic go with these handson lessons and challenges code httpsgithubcombootdotdevfcclearngolangassets follow along interactively on bootdev httpsbootdevlearnlearngolang course created by bootdotdev follow lane on twitter httpstwittercomwagslane learn backend on bootdev httpsbootdev need help join the bootdev discord httpsbootdevcommunity documentation used chi router httpsgithubcomgochichi goose migrations httpsgithubcompresslygoose text instructions for the project httpsbootdevbuildblogaggregator contents 00000 intro 00317 ch 1 why write go 02739 ch 2 variables 05111 ch 3 functions 11658 ch 4 structs 13436 ch 5 interfaces 20026 ch 6 errors 22201 ch 7 loops 24821 ch 8 slices 33954 ch 9 maps 40619 ch 10 advanced functions 43103 ch 11 pointers 44802 ch 12 local development 53143 ch 13 channels concurrency 60738 ch 14 mutexes 63056 ch 15 generics 63838 ch 16 quiz 64313 p1 rss aggregator project 65343 p2 chi router 71137 p3 postgres database 73910 p4 authentication w api keys 81828 p5 many to many relationships 83913 p6 aggregation worker 90528 p7 viewing blog posts thanks to our champion and sponsor supporters davthecoder jediorsith agustn kussrow nattira maneerat heather wcislo serhiy kalinets justin hual otis morgan learn to code for free and get a developer job httpswwwfreecodecamporg read hundreds of articles on programming httpsfreecodecamporgnews support for this channel comes from our friends at scrimba the coding platform thats reinvented interactive learning httpsscrimbacomfreecodecamp,freeCodeCamp.org,27,2023-05-11 13:41:01+00:00,PT9H32M48S,1270394,23500,524,11300000,918891399,1979,2014-12-16 21:18:48+00:00,False,"[Knowledge, Technology]","[golang, programming, go]",34368,0.018498,0.000412,0.018911
3,full golang tutorial learn go by building a todolist app,a complete go crash course for beginners to learn the core go concepts by writing a simple todolist application r e s o u r c e s git repo httpsgitlabcomtwnyoutubegolangcrashcourse http url ip addresses explained it beginners course lecture httpstechworldwithnanateachablecomcoursesitbeginnerscourselectures44206531 golang full course httpsyoutubeyyuhqiec83i thanks jetbrains for making this video possible interested in trying goland httpsjbggcheckgoland jetbrains has given me a 100 discount code use gowithnana2024 to enjoy the full goland ide for 6 months completely free redeem it here

In [ ]:
# Lưu kêt quả đã làm sạch
df.to_csv("..\\data\\processed\\YoutubeCrawl_cleaned.csv", index=False)